# 90_pipelines / 91__full_pipeline

**Entry point for the Databricks Job.**  
Runs the full ingestion → transformation → analysis pipeline in sequence.

```
favorites.json           editar favoritos (00_config/favorites.json en el repo)
      ↓
02__tickers_master       build main.config.tickers
      ↓
11__fetch_sec_xbrl       fetch from SEC API → financials_raw
      ↓
12__fetch_market_data    fetch from Yahoo Finance → market_data
      ↓
21__clean_and_merge      deduplicate → MERGE into financials
      ↓
22__derived_metrics      margins, FCF, YoY, leverage, valuation ratios
      ↓
31__company_analysis     validation queries
```

## 0. Pipeline parameters

Override at runtime via Databricks Job parameters:  
`{"tickers_override": "AAPL,TSLA", "run_optimization": "false"}`

In [0]:
tickers_override = dbutils.widgets.get("tickers_override") if dbutils.widgets.getAll() else ""
run_optimization = dbutils.widgets.get("run_optimization") if dbutils.widgets.getAll() else "false"

dbutils.jobs.taskValues.set("tickers_override", tickers_override)
dbutils.jobs.taskValues.set("run_optimization", run_optimization)

## 1. Load config

In [0]:
%run "/Workspace/Users/al.lopez.moreira@gmail.com/fundamentals_databricks_pj/FA PJ (Basic)/00_config/01__tickers"

In [0]:
from datetime import datetime

if tickers_override:
    ACTIVE_TICKERS = [t.strip() for t in tickers_override.split(",") if t.strip()]
    print(f"✓ Override mode — {len(ACTIVE_TICKERS)} tickers: {ACTIVE_TICKERS}")
else:
    tickers_df = spark.table(f"{CATALOG}.config.tickers")
    ACTIVE_TICKERS = [row.ticker for row in tickers_df.select("ticker").collect()]
    print(f"✓ Config loaded — {len(ACTIVE_TICKERS)} tickers from main.config.tickers")

if not ACTIVE_TICKERS:
    raise ValueError("No tickers found — run 02__tickers_master first.")

pipeline_start = datetime.utcnow()
print(f"Pipeline started at {pipeline_start.isoformat()} UTC")

## 2. Ingestion — fetch from SEC EDGAR
`financials_raw` — append-only audit log

In [0]:
print("=" * 55)
print("STEP 1 / 5 — SEC Ingestion")
print("=" * 55)

In [0]:
%run "/Workspace/Users/al.lopez.moreira@gmail.com/fundamentals_databricks_pj/FA PJ (Basic)/10_ingestion/11__fetch_sec_xbrl"

## 3. Ingestion — fetch market data from Yahoo Finance
`market_data` — year-end prices + market cap

In [0]:
print("=" * 55)
print("STEP 2 / 5 — Market Data")
print("=" * 55)

In [0]:
%run "/Workspace/Users/al.lopez.moreira@gmail.com/fundamentals_databricks_pj/FA PJ (Basic)/10_ingestion/12__fetch_market_data"

## 4. Clean & merge → fact table
`financials` — deduplicated, clean long-format fact table

In [0]:
print("=" * 55)
print("STEP 3 / 5 — Clean & Merge")
print("=" * 55)

In [0]:
%run "/Workspace/Users/al.lopez.moreira@gmail.com/fundamentals_databricks_pj/FA PJ (Basic)/20_transformation/21__clean_and_merge"

## 5. Derived metrics
`financials_metrics` — margins, FCF, YoY growth, leverage, valuation ratios

In [0]:
print("=" * 55)
print("STEP 4 / 5 — Derived Metrics")
print("=" * 55)

In [0]:
%run "/Workspace/Users/al.lopez.moreira@gmail.com/fundamentals_databricks_pj/FA PJ (Basic)/20_transformation/22__derived_metrics"

## 6. Analysis
Runs analysis queries — useful for validation after pipeline runs

In [0]:
print("=" * 55)
print("STEP 5 / 5 — Analysis")
print("=" * 55)

In [0]:
%run "/Workspace/Users/al.lopez.moreira@gmail.com/fundamentals_databricks_pj/FA PJ (Basic)/30_analysis/31__company_analysis"

## 7. Pipeline summary

In [0]:
pipeline_end = datetime.utcnow()
duration     = (pipeline_end - pipeline_start).total_seconds()

print(f"\n{'='*55}")
print(f"  Pipeline completed ✓")
print(f"{'='*55}")
print(f"  Started  : {pipeline_start.isoformat()} UTC")
print(f"  Finished : {pipeline_end.isoformat()} UTC")
print(f"  Duration : {duration:.1f}s ({duration/60:.1f} min)")
print(f"  Tickers  : {len(ACTIVE_TICKERS):,}")
print()

for tbl in ["financials_raw", "financials", "market_data", "financials_metrics"]:
    try:
        n = spark.table(f"{CATALOG}.{SCHEMA}.{tbl}").count()
        print(f"  {CATALOG}.{SCHEMA}.{tbl}: {n:,} rows")
    except Exception:
        pass

## 8. Optional: optimize Delta tables
Set `run_optimization=true` in Job params to enable. Run at most once a week.

In [0]:
if run_optimization.lower() == "true":
    print("Running OPTIMIZE + VACUUM...")
    for tbl in ["financials", "market_data", "financials_metrics"]:
        full = f"{CATALOG}.{SCHEMA}.{tbl}"
        spark.sql(f"OPTIMIZE {full}")
        spark.sql(f"VACUUM {full} RETAIN 168 HOURS")
        print(f"  ✓ {full}")
else:
    print("Optimization skipped (pass run_optimization=true to enable)")